# AI Trainer: QLoRA fine-tune of the coach brain (Kaggle)
**Before running:** right sidebar -> *Session options* -> **Accelerator: GPU T4 x2 (or P100)** and **Internet: On** (needs a phone-verified Kaggle account).
Edit `REPO_URL` below, then *Run All*. Takes roughly 15-30 minutes.

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"   # <- your repo (public, or use a token)
!rm -rf /kaggle/working/repo && git clone --depth 1 {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo/training
!pip install -q -U peft bitsandbytes accelerate transformers mlflow scikit-learn

## 1. Data
Parses the prompt + knowledge base straight from `web/agent.js`, simulates reps with NumPy. Add `--logs` with your downloaded session logs to mix in real states.

In [ ]:
!python gen_data.py

## 2. Baseline: the untuned base model (so the fine-tune has a number to beat)

In [ ]:
!python eval.py --predictor hf --model HuggingFaceTB/SmolLM2-360M-Instruct --name base

## 3. QLoRA fine-tune (4-bit NF4 + LoRA r=16, loss on the JSON answer only)

In [ ]:
!python finetune_qlora.py --epochs 2

## 4. Evaluate the fine-tuned model on the frozen test set (+ log adapter to MLflow and register a version)

In [ ]:
!python eval.py --predictor hf --model HuggingFaceTB/SmolLM2-360M-Instruct --adapter out/adapter --name candidate --register trainer-coach

## 5. Compare against production (same gate CI runs)

In [ ]:
!python gate.py --candidate metrics/candidate.json --production metrics/production.json --min-valid 0.95

## 6. (Optional) merge + push to the Hugging Face Hub for ONNX conversion
Add a Kaggle secret named `HF_TOKEN` (Add-ons -> Secrets), then set `HF_REPO`.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO = "YOUR_HF_USER/trainer-coach-smollm2-360m"
!python merge_export.py --adapter out/adapter --out out/merged --push {HF_REPO}

## 7. Package results
Download `results.zip` from the *Output* tab, then commit `metrics/candidate.json` to your repo.

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip training/metrics training/out/adapter mlflow.db && ls -lh /kaggle/working/results.zip